# Human Pose Estimation — OpenPose BODY_25 via `cv2.dnn`

Estimating human body pose with the BODY_25 model: heatmaps, Part Affinity
Fields, and both the bottom-up and the top-down pipeline.

All paths are relative to the project root, so Jupyter has to be started from
there. Kernel: **Python 3.11 (Computer Vision venv)**.

## What the network returns

BODY_25 is a fully convolutional Caffe graph. It takes an image and returns a
tensor of shape `(1, 78, H/8, W/8)`. The stride of 8 comes from four poolings in
the VGG-like backbone: every output cell covers an 8x8 square of the input.

Those 78 channels are three different things packed into one tensor:

| channels | what it is | how to read it |
|---|---|---|
| 0–24 | heatmaps of the 25 joints | value = probability that the joint is here |
| 25 | background | not needed, but it occupies a channel |
| 26–77 | PAF (Part Affinity Fields) | 26 limbs x 2 channels: the x and y component of a unit vector along the limb |

Heatmaps answer *where the joints are*, PAFs answer *which joints belong to the
same person*. The second question is why PAFs exist at all: with three people in
frame, the left-wrist heatmap has three peaks, and nothing else tells you whose
is whose.

Two approaches to keep apart:

- **bottom-up** — run the whole frame once, find every joint, group them with
  PAFs. Runtime does not depend on the number of people.
- **top-down** — run a person detector first, then pose on each crop. The crop is
  stretched to the network input, so small people "grow"; you pay per person.

---

## Block 1. Imports

In [ ]:
import os
import cv2
import numpy as np
import urllib.request          # a bare `import urllib` does NOT give you this submodule
import matplotlib.pyplot as plt
import time

%matplotlib inline

---

## Block 2. Downloading the model, with a size check

Pulls the architecture (`pose_deploy.prototxt`, ~42 KB) and the weights
(`pose_iter_584000.caffemodel`, exactly 104,715,850 bytes) into
`models/pose_body25/`.

Why not a one-line `urlretrieve`:

1. The official CMU host (`posefs1.perception.cs.cmu.edu`) stopped resolving
   long ago, which kills every tutorial that uses it. The architecture comes
   from CMU's GitHub, the weights from a Hugging Face mirror.
2. 100 MB over a flaky link gets cut halfway. On a broken connection
   `urlretrieve` leaves a **truncated file that looks perfectly normal**:
   `os.path.isfile()` says `True`, the next run skips the download, and
   `readNetFromCaffe` then dies with an unreadable protobuf error.

So the function below resumes with a `Range` header and verifies the final size
against a known number instead of trusting that the file exists.

In [ ]:
MODEL_DIR = "models/pose_body25"
protoFile = os.path.join(MODEL_DIR, "pose_deploy.prototxt")
weightsFile = os.path.join(MODEL_DIR, "pose_iter_584000.caffemodel")

PROTO_URL = ("https://raw.githubusercontent.com/CMU-Perceptual-Computing-Lab/"
             "openpose/master/models/pose/body_25/pose_deploy.prototxt")
WEIGHTS_URL = "https://huggingface.co/dylanholmes/openpose-caffemodels/resolve/main/body25.caffemodel"
WEIGHTS_SIZE = 104_715_850     # anything smaller means the download was cut short


def download(url, dst, expected_size=None, attempts=10, timeout=60):
    """Download `url` into `dst`, resuming after a dropped connection."""
    os.makedirs(os.path.dirname(dst) or ".", exist_ok=True)

    for attempt in range(1, attempts + 1):
        have = os.path.getsize(dst) if os.path.isfile(dst) else 0
        if expected_size and have >= expected_size:
            return True
        if expected_size is None and have:
            return True

        # Ask the server to continue from the byte the last attempt stopped at.
        headers = {"Range": f"bytes={have}-"} if have else {}
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=timeout) as resp:
                # 206 = the server honoured Range, so appending is safe.
                # 200 = it ignored Range and resends from byte 0, so appending
                #       would corrupt the file - reset and overwrite instead.
                mode = "ab" if (have and resp.status == 206) else "wb"
                if mode == "wb":
                    have = 0
                total = have + int(resp.headers.get("Content-Length", 0))
                reported = have
                with open(dst, mode) as f:
                    while True:
                        chunk = resp.read(1 << 20)
                        if not chunk:
                            break
                        f.write(chunk)
                        have += len(chunk)
                        # Report every 10 MB, not every chunk: "\r" does not
                        # erase the line in Jupyter the way it does in a
                        # terminal, so each print would become its own line.
                        if have - reported >= 10 << 20:
                            reported = have
                            print(f"  {os.path.basename(dst)}: {have/1e6:6.1f} / {total/1e6:.1f} MB")
        except Exception as exc:
            # A dropped connection is the normal case here, not a failure.
            print(f"  attempt {attempt} interrupted: {exc}")
            time.sleep(2)

    # Trust the size, not the mere existence of the file.
    return os.path.isfile(dst) and (not expected_size or os.path.getsize(dst) >= expected_size)


download(PROTO_URL, protoFile)
download(WEIGHTS_URL, weightsFile, WEIGHTS_SIZE)
print("prototxt:", os.path.getsize(protoFile), "bytes")
print("weights :", os.path.getsize(weightsFile), "bytes  (expected", WEIGHTS_SIZE, ")")

---

## Block 3. Loading the network

Two things worth knowing here. The argument order of `readNetFromCaffe` is
**prototxt first, weights second** (`readNetFromTensorflow` is the other way
round) — swap them and you get a protobuf error that looks exactly like the one
a truncated file gives.

And CPU is not a compromise here: there is no CUDA on a Mac, `cv2.dnn` does not
use Metal, and for this network the OpenCL target is usually slower than CPU.

In [ ]:
net = cv2.dnn.readNetFromCaffe(protoFile, weightsFile)     # prototxt first, then weights
net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

print("layers:", len(net.getLayerNames()))                 # 261

---

## Block 4. BODY_25 constants

Three lists, and all three have to agree with each other.

**Joints** — the index is the heatmap channel:

```
0 Nose      5 LShoulder  10 RKnee    15 REye   20 LSmallToe
1 Neck      6 LElbow     11 RAnkle   16 LEye   21 LHeel
2 RShoulder 7 LWrist     12 LHip     17 REar   22 RBigToe
3 RElbow    8 MidHip     13 LKnee    18 LEar   23 RSmallToe
4 RWrist    9 RHip       14 LAnkle   19 LBigToe 24 RHeel
```

R/L are from the point of view of **the person in the frame**, not the viewer.

`POSE_PAIRS` holds the 26 limbs the network computes PAFs for, and `MAP_IDX`
holds the two PAF channels of each limb relative to `PAF_OFFSET`. Both come from
`poseParameters.cpp` in OpenPose and their order is locked together:
`MAP_IDX[k]` are the channels for `POSE_PAIRS[k]`. Reorder either one and the
skeleton will connect a knee to an ear — a silent failure, with no exception.

In [ ]:
# Index = heatmap channel. R/L are from the point of view of the person in frame.
BODY_PARTS = ["Nose", "Neck", "RShoulder", "RElbow", "RWrist", "LShoulder",
              "LElbow", "LWrist", "MidHip", "RHip", "RKnee", "RAnkle", "LHip",
              "LKnee", "LAnkle", "REye", "LEye", "REar", "LEar", "LBigToe",
              "LSmallToe", "LHeel", "RBigToe", "RSmallToe", "RHeel"]

N_POINTS = len(BODY_PARTS)     # 25
PAF_OFFSET = N_POINTS + 1      # 0-24 are joints, 25 is background, 26+ are PAFs

In [ ]:
# The order of these two lists is locked together: MAP_IDX[k] are the PAF
# channels of POSE_PAIRS[k]. Reorder one of them and the skeleton connects a
# knee to an ear - silently, without raising anything.
POSE_PAIRS = [[1, 8], [1, 2], [1, 5], [2, 3], [3, 4], [5, 6], [6, 7], [8, 9],
              [9, 10], [10, 11], [8, 12], [12, 13], [13, 14], [1, 0], [0, 15],
              [15, 17], [0, 16], [16, 18], [2, 17], [5, 18], [14, 19], [19, 20],
              [14, 21], [11, 22], [22, 23], [11, 24]]

MAP_IDX = [[0, 1], [14, 15], [22, 23], [16, 17], [18, 19], [24, 25], [26, 27],
           [6, 7], [2, 3], [4, 5], [8, 9], [10, 11], [12, 13], [30, 31],
           [32, 33], [36, 37], [34, 35], [38, 39], [20, 21], [28, 29], [40, 41],
           [42, 43], [44, 45], [46, 47], [48, 49], [50, 51]]

# Pairs [2,17] and [5,18] (shoulder-to-ear): useful for grouping people, but on
# the drawing they strike a line straight across the face.
SKIP_RENDER = {18, 19}

# One BGR colour per limb, spread evenly around the hue circle.
COLORS = [tuple(int(v) for v in cv2.cvtColor(
              np.uint8([[[i * 180 // len(POSE_PAIRS), 255, 255]]]), cv2.COLOR_HSV2BGR)[0, 0])
          for i in range(len(POSE_PAIRS))]

**Structural check.** The anatomy check comes in block 7, once `draw_pose`
exists — but this one is free and catches half the mistakes right here. The
second assert is the useful one: a duplicated or missing PAF channel fails now
instead of showing up as a crooked skeleton three blocks later.

In [ ]:
assert len(POSE_PAIRS) == len(MAP_IDX) == 26
assert sorted(c for pair in MAP_IDX for c in pair) == list(range(52))   # every PAF channel exactly once
assert max(j for pair in POSE_PAIRS for j in pair) == N_POINTS - 1      # no joint index above 24
assert len(COLORS) == len(POSE_PAIRS)
print("constants OK")

A small helper used everywhere below — matplotlib expects RGB, OpenCV gives BGR.

In [ ]:
def show(im, title="", figsize=(11, 8)):
    plt.figure(figsize=figsize)
    plt.imshow(im[:, :, ::-1])      # BGR -> RGB for matplotlib
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

---

## Block 5. One forward pass

Every argument of `blobFromImage` has a reason:

- `1/255.0` — the model was trained on inputs in the range [0, 1];
- `(in_width, in_height)` — the width follows the frame's aspect ratio and is
  rounded to a multiple of 8. Break the ratio and the person is squeezed
  horizontally, which costs accuracy; skip the rounding and OpenCV pads the size
  itself, shifting the maps by half a cell;
- `(0, 0, 0)` — no mean subtraction, the model does not expect it;
- **`swapRB=False`** — the Caffe model was trained on BGR and `cv2.imread`
  already returns BGR. Set it to `True` (as TensorFlow models want) and the
  network still finds *something*, just worse and less reliably. A classic
  silent bug;
- `crop=False` — otherwise OpenCV centre-crops the frame instead of resizing it.

The last line stretches all 78 maps back to frame size, so peak coordinates are
already in original pixels and nothing below has to multiply by `w/out_w`. The
price is memory: 78 float maps at full frame size.

In [ ]:
def run_pose(net, im, in_height=368):
    h, w = im.shape[:2]
    # Width follows the frame aspect ratio, rounded to a multiple of the
    # network stride (8).
    in_width = max(8, int(round(in_height * w / h / 8)) * 8)

    blob = cv2.dnn.blobFromImage(im, 1 / 255.0, (in_width, in_height),
                                 (0, 0, 0), swapRB=False, crop=False)

    net.setInput(blob)
    out = net.forward()             # (1, 78, in_height/8, in_width/8)

    # Stretch every map back to frame size, so peaks are already in the pixel
    # coordinates of the original image.
    return np.stack([cv2.resize(out[0, i], (w, h)) for i in range(out.shape[1])])

**Visual check.** Expect `~0.7–1.2 s` and `maps.shape == (78, 342, 548)`.

Nothing is drawn on the frame yet — `run_pose` does not modify `im`, and
`draw_pose` only appears in block 7. What this cell proves is that the maps come
back in frame pixels and that the image itself loaded correctly.

In [ ]:
im = cv2.imread("data/images/messi5.jpg")

t = time.time()
maps = run_pose(net, im)
print(f"{time.time() - t:.2f} s   im.shape={im.shape}   maps.shape={maps.shape}")

# The real check here. Without the cv2.resize inside run_pose the shape would be
# (78, 46, 69), and every coordinate computed below would be off by a factor of 8.
assert maps.shape == (78,) + im.shape[:2]

# Plain input frame - no lines yet. This only confirms the image decoded and the
# BGR->RGB swap in show() is right (the grass has to look green, not red).
show(im, "input frame")

In [ ]:
def show_map (im, m, title="", alpha=0.6):
    plt.figure(figsize=(11,8))
    plt.imshow(im[:,:,::-1])
    plt.imshow(m,alpha=alpha,cmap="jet")
    plt.axis("off")
    plt.title(f"{title} max={m.max():.2f}")
    plt.show()


In [ ]:
maps = run_pose(net, im)

for name in ["Nose","RWrist","LAnkle"]:
    i = BODY_PARTS.index(name)
    show_map(im,maps[i], f"heatmap {name}")

In [ ]:
k = POSE_PAIRS.index([3,4])
paf = np.hypot(maps[PAF_OFFSET + MAP_IDX[k][0]],
              maps[PAF_OFFSET + MAP_IDX[k][1]])
show_map(im,paf,"PAF RElbow->RWrist")

In [ ]:
for pair in ([3,4],[10,11],[1,8]):
    k = POSE_PAIRS.index(pair)
    paf = np.hypot(maps[PAF_OFFSET + MAP_IDX[k][0]],
                  maps[PAF_OFFSET + MAP_IDX[k][1]])
    show_map(im,paf,f"PAF {BODY_PARTS[pair[0]]} -> {BODY_PARTS[pair[1]]}")

In [ ]:
def keypoints_single(maps, threshold=0.1):
    points = []
    for i in range(N_POINTS):
        _, conf, _, loc = cv2.minMaxLoc(maps[i])
        points.append((loc[0], loc[1], conf) if conf > threshold else None)
    return points

In [ ]:
points = keypoints_single(run_pose(net,im))
missing = [BODY_PARTS[i] for i, p in enumerate(points) if p is None]
print(f"found {N_POINTS - len(missing)}/{N_POINTS}, lost: {missing}")

In [ ]:
def get_keypoints(prob_map, threshold=0.1):
    smooth = cv2.GaussianBlur(prob_map, (3,3),0,0)
    mask = np.uint8(smooth > threshold)

    keypoints = []
    contours, _ = cv2.findContours(mask, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        blob_mask = cv2.fillConvexPoly(np.zeros(mask.shape), cnt, 1)
        _,_,_, loc = cv2.minMaxLoc(smooth * blob_mask)
        keypoints.append(loc + (prob_map[loc[1], loc[0]],))
    return keypoints

In [ ]:
def detect_keypoints(maps, threshold=0.1):
    detected, keypoints_list, next_id = [], [], 0
    for i in range(N_POINTS):
        with_id = []
        for p in get_keypoints(maps[i],threshold):
            keypoints_list.append(p[:3])
            with_id.append(p+(next_id,))
            next_id += 1
        detected.append(with_id)
    return detected, np.array(keypoints_list).reshape(-1,3)

In [ ]:
image_solders = cv2.imread("data/images/solders.jpg")
maps = run_pose(net, im)
detected, keypoints_list = detect_keypoints(maps)

print("Candidates amount:", len(keypoints_list))
for i, cand in enumerate(detected):
    if len(cand) > 1:
        print(f" {BODY_PARTS[i]:10} x {len(cand)}")

vis = im.copy()
for cand in detected: 
    for x, y, conf, _id in cand:
        cv2.circle(vis,(x,y), 4, (0,255,255), -1)
show(vis, f"{len(keypoints_list)} canditates")

In [ ]:
def get_valid_pairs(maps, detected, n_interp=10, paf_threshold=0.1, conf_threshold=0.7):
    """Greedily match joint candidates into limbs, one limb type at a time.

    Returns a list of length len(POSE_PAIRS); element k is an (N, 3) array of
    [id_a, id_b, score] rows - the connections accepted for POSE_PAIRS[k].
    The ids are the ones detect_keypoints handed out, not coordinates.
    """
    valid_pairs = []

    for k in range(len(POSE_PAIRS)):
        paf_x = maps[PAF_OFFSET + MAP_IDX[k][0]]
        paf_y = maps[PAF_OFFSET + MAP_IDX[k][1]]

        cand_a = detected[POSE_PAIRS[k][0]]
        cand_b = detected[POSE_PAIRS[k][1]]

        # Nothing on one end - no limbs of this type. The empty entry still has
        # to be appended, or index k stops lining up with POSE_PAIRS.
        if not cand_a or not cand_b:
            valid_pairs.append(np.empty((0, 3)))
            continue

        pairs, used_b = [], set()

        for pa in cand_a:
            # Reset per candidate A, otherwise the first match blocks the rest.
            best_score, best_j = -1.0, -1

            for j, pb in enumerate(cand_b):
                if j in used_b:            # one B belongs to one person only
                    continue

                d = np.subtract(pb[:2], pa[:2])
                norm = np.linalg.norm(d)
                if norm < 1e-6:            # two detections on the same pixel
                    continue
                d = d / norm

                # Sample the PAF along the straight line from A to B.
                xs = np.linspace(pa[0], pb[0], n_interp)
                ys = np.linspace(pa[1], pb[1], n_interp)
                vectors = np.array([[paf_x[int(round(y)), int(round(x))],
                                     paf_y[int(round(y)), int(round(x))]]
                                    for x, y in zip(xs, ys)])

                # Projection of each sampled PAF vector onto the A->B direction.
                scores = vectors @ d
                mean = scores.mean()

                # Two different gates: paf_threshold filters a single sample,
                # conf_threshold demands that most samples pass it.
                if (np.sum(scores > paf_threshold) / n_interp) > conf_threshold \
                        and mean > best_score:
                    best_score, best_j = mean, j

            if best_j != -1:
                used_b.add(best_j)
                pairs.append([pa[3], cand_b[best_j][3], best_score])

        valid_pairs.append(np.array(pairs).reshape(-1, 3))

    return valid_pairs

In [ ]:
valid_pairs = get_valid_pairs(maps, detected)

assert len(valid_pairs) == len(POSE_PAIRS)      # k must still index POSE_PAIRS
print(f"accepted limbs: {sum(len(v) for v in valid_pairs)} of {len(keypoints_list)} candidates\n")

for k, vp in enumerate(valid_pairs):
    if len(vp):
        a, b = POSE_PAIRS[k]
        print(f"  {BODY_PARTS[a]:>10} -> {BODY_PARTS[b]:<11} {len(vp)}  "
              f"scores={np.round(vp[:, 2], 2).tolist()}")

In [ ]:
valid_pairs = get_valid_pairs(maps, detected)
print("Valid pairs", sum(len(v) for v in valid_pairs))

vis = im.copy()
for k , pairs in enumerate(valid_pairs):
    if k in SKIP_RENDER:
        continue
    for id_a, id_b, score in pairs:
        xa, ya = keypoints_list[int(id_a),:2]
        xb, yb = keypoints_list[int(id_b), :2]
        cv2.line(vis, (int(xa), int(ya)), (int(xb), int(yb)), COLORS[k],2)
show(vis, "valid pairs before grouping")

In [ ]:
def group_people(valid_pairs, keypoints_list):
    people = np.empty((0, N_POINTS + 1)) 

    for k, (a,b) in enumerate(POSE_PAIRS):
        for id_a, id_b, score in valid_pairs[k]:
            found = next((i for i, person in enumerate(people) if person[a] == id_a), -1)
            if found != -1 :
                people[found][b] = id_b
                people[found][-1] += keypoints_list[int(id_b), 2] + score 
            elif k < 17:
                row = np.full(N_POINTS +1, -1.0)
                row[a] , row[b] = id_a, id_b
                row[-1] = keypoints_list[[int(id_a),int(id_b)], 2].sum() + score
                people = np.vstack([people, row])

    return people 

In [ ]:
def draw_people(im, people, keypoints_list, thickness=2, radius=4):
    """Draw one skeleton per row of `people` on a copy of `im`.

    Colour comes from COLORS[k], the limb index - so the same body part is the
    same colour on every person. That is what makes a broken POSE_PAIRS/MAP_IDX
    pairing visible: a forearm in the colour of a shin means the lists slipped.
    """
    vis = im.copy()

    for person in people:
        for k, (a, b) in enumerate(POSE_PAIRS):
            if k in SKIP_RENDER:              # shoulder-to-ear: a line across the face
                continue

            # -1 means this person has no candidate for that joint.
            id_a, id_b = int(person[a]), int(person[b])
            if id_a < 0 or id_b < 0:
                continue

            xa, ya = keypoints_list[id_a, :2].astype(int)
            xb, yb = keypoints_list[id_b, :2].astype(int)
            cv2.line(vis, (xa, ya), (xb, yb), COLORS[k], thickness, cv2.LINE_AA)

        # Joints last, so the dots sit on top of the lines, not under them.
        for i in range(N_POINTS):
            idx = int(person[i])
            if idx >= 0:
                x, y = keypoints_list[idx, :2].astype(int)
                cv2.circle(vis, (x, y), radius, (255, 255, 255), -1, cv2.LINE_AA)

    return vis

In [ ]:
t = time.time()
maps = run_pose(net, image_solders)
detected, keypoints_list = detect_keypoints(maps)
valid_pairs = get_valid_pairs(maps, detected)
people = group_people(valid_pairs, keypoints_list)

MIN_JOINTS = 8
n_joints = (people[:, :-1] >= 0).sum(axis=1)
strong = people[n_joints >= MIN_JOINTS]

print(f"{time.time() - t:.2f}s  candidates: {len(keypoints_list)}  "
      f"rows: {len(people)} -> {len(strong)}")

show(draw_people(image_solders, strong, keypoints_list), "bottom-up")

In [ ]:
image_basketball = cv2.imread("data/images/basketball1.png")

for in_h in (256, 368, 512):
    t = time.time()
    maps = run_pose(net, image_basketball, in_height = in_h)
    detected, keypoints_list = detect_keypoints(maps)
    valid_pairs = get_valid_pairs(maps, detected)
    people = group_people(valid_pairs, keypoints_list)
    show(draw_people(image_basketball, people, keypoints_list),
         f"in_height={in_h} {time.time()-t:.2f} with skelets: {len(people)}"
        )

In [ ]:
def pose_top_down(im, in_height=368, det_threshold=0.4, pad=0.15, draw_boxes=True):
    out = im.copy()
    for x1, y1, x2, y2, score in detect_persons(im, det_threshold):
        m = int(pad * max(x2 - x1, y2 - y1))
        cx1, cy1 = max(0, x1-m),max(0, y1-m)
        cx2, cy2 = min(im.shape[1], x2+m), min(im.shape[0], y2 + m)
        crop = im[cy1:cy2,cx1:cx2]
        if crop.size == 0:
            continue

        points = keypoints_single(run_pose(net, crop, in_height))
        points = [None if p is None else (p[0] + cx1, p[1] + cy1, p[2]) for p in points]

        scale = max(1,(y2-y1)//60)
        if draw_boxes:
            cv2.rectangle(out, (x1, y1), (x2, y2), (0,255,255), 1)
        out = draw_pose(out, points, radius=scale + 1, thickness=scale)
        return out

In [ ]:
cap = cv2.VideoCapture("videos/vtest.mp4")
cap.set(cv2.CAP_PROP_POS_FRAMES, 300)
ok, frame = cap.read()
cap.release()
assert ok, "frame wasn't captured"

t = time.time()
maps = run_pose(net, frame)
detected, keypoints_list = detect_keypoints(maps)
valid_pairs = get_valid_pairs(maps, detected)
people = group_people(valid_pairs, keypoints_list)
show(draw_people(frame, people, keypoints_list), f"bottom-up {time.time() - t:.2f} s")

In [ ]:
def draw_pose(im, points, radius=4, thickness=2):
    """Draw one skeleton from a keypoints_single() list onto a copy of `im`.

    `points[i]` is either (x, y, conf) or None - the top-down path needs the
    None case constantly, because a crop often cuts off feet or an ear.
    """
    vis = im.copy()

    for k, (a, b) in enumerate(POSE_PAIRS):
        if k in SKIP_RENDER:
            continue
        if points[a] is None or points[b] is None:
            continue
        pa = (int(points[a][0]), int(points[a][1]))
        pb = (int(points[b][0]), int(points[b][1]))
        cv2.line(vis, pa, pb, COLORS[k], thickness, cv2.LINE_AA)

    for p in points:
        if p is not None:
            cv2.circle(vis, (int(p[0]), int(p[1])), radius, (255, 255, 255), -1, cv2.LINE_AA)

    return vis

In [ ]:
DET_DIR = "models/ssd_mobilenet_v2_coco_2018_03_29"

# A SECOND network. Do not name it `net` - that one is BODY_25, and run_pose
# takes it as an argument. Overwrite `net` and every pose cell above breaks.
det_net = cv2.dnn.readNetFromTensorflow(
    os.path.join(DET_DIR, "frozen_inference_graph.pb"),
    os.path.join(DET_DIR, "ssd_mobilenet_v2_coco_2018_03_29.pbtxt"))


def detect_persons(im, threshold=0.4):
    """Return [(x1, y1, x2, y2, score), ...] for people only."""
    h, w = im.shape[:2]

    # swapRB=True here, unlike run_pose: this one is a TensorFlow model trained
    # on RGB, while BODY_25 is Caffe and wants BGR.
    blob = cv2.dnn.blobFromImage(im, 1.0, (300, 300), (0, 0, 0), swapRB=True, crop=False)
    det_net.setInput(blob)
    out = det_net.forward()          # (1, 1, N, 7)

    boxes = []
    for i in range(out.shape[2]):
        if int(out[0, 0, i, 1]) != 1:        # COCO class 1 = person
            continue
        score = float(out[0, 0, i, 2])
        if score < threshold:
            continue
        # Columns 3..6 are normalised to [0, 1] - multiply by frame size.
        x1, y1 = int(out[0, 0, i, 3] * w), int(out[0, 0, i, 4] * h)
        x2, y2 = int(out[0, 0, i, 5] * w), int(out[0, 0, i, 6] * h)
        boxes.append((max(0, x1), max(0, y1), min(w, x2), min(h, y2), score))

    return boxes

In [ ]:
src, dst = "outputs/pose-vtest.mp4", "outputs/pose-vtest-h264.mp4"

cap = cv2.VideoCapture(src)
fps = cap.get(cv2.CAP_PROP_FPS)
size = (int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))

# avc1 = H.264. This is the only reason the old file would not play.
out = cv2.VideoWriter(dst, cv2.VideoWriter_fourcc(*"avc1"), fps, size)
assert out.isOpened(), "no H.264 encoder in this OpenCV build"

while True:
    ok, frame = cap.read()
    if not ok:
        break
    out.write(frame)

cap.release()
out.release()
print(f"{os.path.getsize(dst) / 1e6:.2f} MB")

Video(dst, embed=True, width=720)

In [ ]:
cap = cv2.VideoCapture(0)
assert cap.isOpened(), "camera did not open"

# The first frames come out almost black: the Mac camera needs a moment to
# settle its auto-exposure. Read a few and throw them away.
for _ in range(10):
    cap.read()

ok, frame = cap.read()
cap.release()
assert ok, "camera opened but gave no frame"
print("Camera:", ok, frame.shape)

frame = cv2.flip(frame, 1)
show(draw_pose(frame, keypoints_single(run_pose(net, frame, in_height=192))),
     "one frame from camera")